# AKE Lab 5 · Zeitreihen: Day-ahead-Lastprognose — Starter-Notebook

**Angewandte KI & KI-Engineering · Block B – Spezialisierte Anwendungsgebiete · Woche 5**
Bezug: Foliensatz *Zeitreihen & Reinforcement Learning*, Abschnitte 1–5 · Lektüre: FPP3 (Hyndman & Athanasopoulos) Kap. 1–3, 5, 8, 9

> **Dieses Notebook ist das komplette Lab – es gibt kein separates Aufgabenblatt.** Jede Aufgabe enthält die Aufgabenstellung,
> die Formeln, die Erwartungswerte und die Leitfragen. Der Code ist ausführlich kommentiert: Lesen Sie die Kommentare
> (Zeilen, die mit `#` beginnen) – sie erklären Zeile für Zeile, was passiert.

**Die Aufgabe:** Ein Übertragungsnetzbetreiber muss jeden Tag die Netzlast des Folgetags Stunde für Stunde vorhersagen
(*Day-ahead*). Sie arbeiten mit echten Daten der Regelzone **Amprion** (Westdeutschland, Stundenwerte 2018 bis Juni 2020, Open
Power System Data). Als Messlatte dient die Day-ahead-Prognose des Netzbetreibers selbst, die im Datensatz mitgeliefert wird.

Sie gehen den Weg aus der Vorlesung: Daten verstehen (A1) → Baselines und Backtest (A2) → statistische Modelle (A3) →
LightGBM und LSTM (A4) → Temporal Fusion Transformer und Modellwahl (A5) → Drift und Anomalien im Corona-Lockdown 2020 (A6).

**Charakter:** Übung in **Blöcken während der Vorlesung** – jeder Block folgt direkt auf den Vorlesungsabschnitt, den er vertieft.
**Ihre Zeit ≈ 80 Min.** in der Sitzung, davon **≈ 3–4 Min. reine Rechenzeit** auf der Referenz (4 Kerne / 8 GB, keine GPU).
Den TFT trainieren Sie nicht – er liegt fertig trainiert in `data/models/tft_amprion/`.

## Organisatorisches

| | |
|---|---|
| **Termin** | Woche 5 (KW 45, 05.11.2026), in sechs Blöcken während der Vorlesung; Besprechung jeweils direkt nach der Arbeitsphase am Beamer |
| **Bezug** | Foliensatz *Zeitreihen & Reinforcement Learning*, Abschn. 1 (Zeitreihen verstehen), 2 (Prognosen bewerten), 3 (ETS, ARIMA, MSTL), 4 (LightGBM, LSTM, TFT), 5 (Anomalien und Drift) |
| **Material** | Ordner `lab05/` im Kursrepository: dieses Notebook, `results_referenz/` (Referenzlogs und TFT-Vorhersagen als Fallback), `lab05_rl_optional.ipynb` (DQN auf CartPole, Vertiefung zu Hause); die Musterlösung folgt nach der Sitzung |
| **Umgebung** | Kursumgebung aktualisieren: `git pull`, danach **`uv sync`** – neu sind `pytorch-forecasting` 1.8.0, `lightning` 2.6.6 und `gymnasium`. Kein Download während des Labs nötig. |
| **Daten** | `data/opsd/amprion_load_2018_2020H1.csv`: 21 888 Stundenwerte (1.1.2018–30.6.2020, UTC), Spalten `load_mw` (Ist-Last), `load_forecast_tso_mw` (Day-ahead-Prognose des Netzbetreibers), `solar_mw`, `wind_mw`; `data/models/tft_amprion/` (TFT-Checkpoint, 1 MB) |

**Fahrplan**

| Aufgabe | Block in der Vorlesung (nach Folie …) | Inhalt | Ihre Zeit | Rechenzeit |
|---|---|---|---|---|
| 0 · Setup | vorab, spätestens zu Beginn | Setup-Zelle, Prüfungen | 2 Min. | Sekunden |
| 1 · Daten erkunden | „Zwischenstand: Zeitreihen verstehen“ | Zeitskalen, Autokorrelation (`CODE`), MSTL-Zerlegung | 15 Min. | Sekunden |
| 2 · Baselines und Backtest | „Die Messlatte“ | Fehlermaße (`CODE`), Baselines (`CODE`), Backtest-Gerüst | 15 Min. | Sekunden |
| 3 · ETS, SARIMA, MSTL | „Zwischenstand: Klassische Modelle“ | Automatik prüfen (`CODE`), SARIMA, MSTL (`CODE`) | 15 Min. | ≈ 1 Min. |
| 4 · LightGBM und LSTM | „Ein LSTM für 24 Stunden“ | Merkmale (`CODE`), LightGBM (`CODE`), LSTM | 15 Min. | ≈ 15 s |
| 5 · TFT und Modellwahl | „Zwischenstand: ML und Deep Learning“ | TFT befragen, Band prüfen (`CODE`), Empfehlung, Endtest | 10 Min. | ≈ 10 s |
| 6 · Drift und Anomalien | „Zwischenstand: Anomalien und Drift“ | Residuen 2020 (`CODE`), Isolation Forest (`CODE`) | 10 Min. | Sekunden |

**Kennzeichnung der Arbeitsarten** (steht am Anfang jeder Teilaufgabe):

| Marke | Was Sie tun |
|---|---|
| `CODE` | eine Lücke selbst füllen (Zeile mit `raise NotImplementedError` durch Ihre Lösung ersetzen) |
| `LIVE` | wird in der Vorlesung gemeinsam geschrieben bzw. vorgeführt – ausführen und mitlesen |
| `AUSFÜHREN` | nur ausführen und die Ausgabe lesen |
| `DENKEN` | Interpretationsfrage in der Markdown-Zelle beantworten |

Wer bei einer Lücke nicht weiterkommt, übernimmt die Zeile aus der Lösung am Beamer – die Aufgaben bauen aufeinander auf.

**Abgabe:** keine. Bringen Sie zur nächsten Sitzung Ihre Vergleichstabelle aus 5.3 und Ihre dreisätzige Modellempfehlung mit.
Für Capstone-Themen mit Zeitreihen (1, 4, 6) ist dieses Notebook eine gute Vorlage: Backtest-Gerüst, Baselines, Merkmale.

## Lernziele

Nach dem Lab können Sie … (Lernziel-Matrix, Woche 5)

- **Erinnern:** die Verfahren naiv, saisonal naiv, ETS, SARIMA, MSTL, LightGBM, LSTM und TFT benennen und einordnen.
- **Verstehen:** erklären, was Trend, Saison, Autokorrelation und ein Backtest mit rollierendem Ursprung sind.
- **Anwenden:** ein Forecasting-Modell leckagefrei trainieren und im Backtest bewerten (MAE, RMSE, MAPE, MASE).
- **Analysieren:** Saisonalität mit ACF und Zerlegung sowie Drift und Anomalien im Residuum analysieren.
- **Bewerten:** eine Modellauswahl begründen – mit Genauigkeit, Aufwand, Unsicherheit und Erklärbarkeit.

## Spielregeln und Hinweise

- **Die Zukunft darf nie ins Training sickern.** Ursprung jeder Prognose ist 23:00 Uhr (UTC) des Vortags; vorhergesagt werden die
  24 Stunden des Folgetags. Jedes Merkmal muss zu diesem Zeitpunkt bekannt sein – also **kein Lag kürzer als 24 h**.
- **Ein Backtest für alle:** 28 Ursprünge vom 7.10. bis 3.11.2019. Alle Modelle werden auf genau diesen 672 Stunden verglichen.
- **Training bis 30.9.2019.** Der **Endtest** (Dezember 2019) wird in Aufgabe 5 **genau einmal** gerechnet – für das Modell, das
  Sie vorher auf dem Backtest ausgewählt haben. Die Daten ab Januar 2020 dienen nur der Drift-Analyse in Aufgabe 6.
- **Kalender in Ortszeit:** Lastprofile folgen der Uhr. Die Hilfsfunktion `kalender()` rechnet UTC in Europe/Berlin um.
- **Reproduzierbarkeit:** feste Zufallszahlen (`seed 0`); jedes Experiment schreibt ein JSON-Log nach `results/`.
- **Fallback:** Läuft der TFT in Aufgabe 5 nicht (z. B. falsche Paketversion), setzen Sie dort `NUTZE_REFERENZ = True`: Dann werden
  die mitgelieferten Vorhersagen aus `results_referenz/tft_backtest.csv` geladen.

## Aufgabe 0 · Setup

`AUSFÜHREN` · Bibliotheken laden, Pfade finden, Daten prüfen, Hilfsfunktionen bereitstellen. Wenn alle Zeilen ✅ zeigen, kann es losgehen.

In [ ]:
# --- Bibliotheken -------------------------------------------------------------------------------------------
import os, sys, time, json, warnings                    # Standardbibliothek: Zeitmessung, JSON-Logs
from pathlib import Path                                # Pfade zu Dateien und Ordnern, unabhaengig vom Betriebssystem
import numpy as np                                      # Rechnen mit Vektoren und Matrizen
import pandas as pd                                     # Tabellen (DataFrame) und Zeitreihen (Series mit Zeitindex)
import matplotlib.pyplot as plt                         # Diagramme
import torch                                            # PyTorch: fuer das LSTM in Aufgabe 4
import lightgbm as lgb                                  # Gradient Boosting fuer Aufgabe 4
from statsforecast import StatsForecast                 # statistische Zeitreihenmodelle (Aufgabe 1 und 3)
from statsforecast.models import AutoETS, ARIMA, MSTL, SeasonalNaive
from sklearn.ensemble import IsolationForest            # Anomalieerkennung (Aufgabe 6)
warnings.filterwarnings("ignore")                       # Bibliotheks-Warnungen ausblenden (Lesbarkeit)
os.environ.setdefault("NIXTLA_ID_AS_COL", "1")          # statsforecast: unique_id als Spalte zurueckgeben

torch.set_num_threads(min(4, os.cpu_count()))           # Referenz: 4 Kerne -> vergleichbare Laufzeiten
np.random.seed(0); torch.manual_seed(0)                 # feste Zufallszahlen -> jeder Lauf liefert dasselbe Ergebnis
plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.spines.top": False, "axes.spines.right": False})

# --- Pruefhelfer --------------------------------------------------------------------------------------------
# check("Name", Bedingung, "Hinweis") gibt ✅ aus, wenn die Bedingung wahr ist, sonst ❌ mit Hinweis.
# Die Bedingung steht meist als `lambda: ...` da: eine kleine Funktion, die erst hier ausgewertet wird.
# So fuehrt ein Fehler in Ihrer Loesung zu einem ❌ mit Fehlermeldung statt zum Abbruch der ganzen Zelle.
CHECKS = {}
def check(name, ok, hint=""):
    try:
        ok = bool(ok() if callable(ok) else ok); msg = hint
    except NotImplementedError as e:                    # die Luecke ist noch nicht gefuellt
        ok, msg = False, f"{e} ist noch offen"
    except Exception as e:                              # irgendein anderer Fehler in der Loesung
        ok, msg = False, f"{type(e).__name__}: {e}" + (f" | {hint}" if hint else "")
    CHECKS[name] = ok
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {msg}")

def nah(a, b, tol):                                     # |a - b| <= tol  (fuer Kontrollwerte mit Rundung)
    return abs(float(a) - float(b)) <= tol

# --- Laufzeitmessung je Aufgabe (Tabelle am Ende; fuer das Lab nicht wichtig) ---------------------------------
RUNTIME, _teil = {}, {"name": "Aufgabe 0", "t0": None}
def abschnitt(name): _teil["name"] = name
def _ake_pre(*a): _teil["t0"] = time.perf_counter()
def _ake_post(*a):
    if _teil["t0"] is not None:
        RUNTIME[_teil["name"]] = RUNTIME.get(_teil["name"], 0.0) + time.perf_counter() - _teil["t0"]
try:
    _ip = get_ipython()
    for ev, fn in (("pre_run_cell", _ake_pre), ("post_run_cell", _ake_post)):
        for old in [f for f in _ip.events.callbacks[ev] if getattr(f, "__name__", "") == fn.__name__]:
            _ip.events.unregister(ev, old)
        _ip.events.register(ev, fn)
except NameError:
    pass

# --- Pfade: REPO_ROOT = erster Elternordner mit pyproject.toml ------------------------------------------------
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden. Bitte Jupyter im Ordner des Repositorys starten und lab05/ oeffnen.")
DATA    = REPO_ROOT / "data"
CSV     = DATA / "opsd" / "amprion_load_2018_2020H1.csv"
TFT_CKPT = DATA / "models" / "tft_amprion" / "tft_amprion.ckpt"
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)   # hierhin schreibt das Notebook seine Logs
REFDIR  = Path("results_referenz")                        # Referenzlogs (Fallback)

def save_log(name, **werte):
    """Schreibt ein JSON-Log nach results/<name>.json."""
    (RESULTS / f"{name}.json").write_text(json.dumps(werte, indent=1, ensure_ascii=False, default=float))

# --- Kalender in Ortszeit -------------------------------------------------------------------------------------
# Gesetzliche Feiertage, die im groessten Teil der Regelzone Amprion gelten (NRW, RP, SL, BW, BY)
FEIERTAGE = pd.to_datetime([
    "2018-01-01", "2018-03-30", "2018-04-02", "2018-05-01", "2018-05-10", "2018-05-21", "2018-05-31",
    "2018-10-03", "2018-11-01", "2018-12-25", "2018-12-26",
    "2019-01-01", "2019-04-19", "2019-04-22", "2019-05-01", "2019-05-30", "2019-06-10", "2019-06-20",
    "2019-10-03", "2019-11-01", "2019-12-25", "2019-12-26",
    "2020-01-01", "2020-04-10", "2020-04-13", "2020-05-01", "2020-05-21", "2020-06-01", "2020-06-11"])

def kalender(idx_utc):
    """Kalendermerkmale in Ortszeit (Europe/Berlin) fuer einen UTC-Zeitindex – Lastprofile folgen der Uhr, nicht UTC."""
    lt = pd.DatetimeIndex(idx_utc).tz_localize("UTC").tz_convert("Europe/Berlin")   # UTC -> deutsche Ortszeit
    tag = lt.tz_localize(None).normalize()                                           # Datum ohne Uhrzeit
    return pd.DataFrame({"stunde": lt.hour, "wochentag": lt.dayofweek, "monat": lt.month,
                         "feiertag": tag.isin(FEIERTAGE).astype(int)}, index=idx_utc)

# --- Feste Zeitraeume (gelten fuer alle Aufgaben) -------------------------------------------------------------
TRAIN_END  = pd.Timestamp("2019-09-30 23:00")                  # letzte Trainingsstunde
URSPRUENGE = pd.date_range("2019-10-07", "2019-11-03", freq="D")   # 28 Prognosetage (je 00:00 UTC bis 23:00 UTC)
BT_INDEX   = pd.date_range("2019-10-07", "2019-11-03 23:00", freq="h")   # die 672 Backtest-Stunden
ENDTEST    = pd.date_range("2019-12-02", "2019-12-29 23:00", freq="h")   # Endtest, genau einmal (Aufgabe 5)

import statsforecast, sklearn
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | lightgbm {lgb.__version__} | "
      f"statsforecast {statsforecast.__version__} | scikit-learn {sklearn.__version__} | Threads {torch.get_num_threads()}")
print(f"REPO_ROOT: {REPO_ROOT}\n")
check("Daten data/opsd/amprion_load_2018_2020H1.csv vorhanden", CSV.exists(), "git pull")
check("TFT-Checkpoint data/models/tft_amprion/ vorhanden", TFT_CKPT.exists(), "git pull")

## Aufgabe 1 · Daten erkunden

**Block nach „Zwischenstand: Zeitreihen verstehen“ · 15 Min.** · Folien Abschnitt 1 · FPP3 Kap. 2–3

### 1.1 · Daten laden

`LIVE` · Die CSV-Datei wird gelesen, die Spalte `ds` wird zum Zeitindex. `y` ist ab jetzt die Ist-Last in MW als Zeitreihe
(pandas-`Series` mit Zeitindex). **Erwartung:** 21 888 Zeilen, keine Lücken, Last zwischen ≈ 12 und ≈ 30 GW.

In [ ]:
abschnitt("Aufgabe 1")
df = pd.read_csv(CSV, parse_dates=["ds"]).set_index("ds")   # Zeitstempel als Index -> Zeitreihe
y = df["load_mw"].astype(float)                             # Ist-Last in MW
print(df.head(3)); print("...")
print(f"{len(df)} Zeilen von {df.index[0]} bis {df.index[-1]} (UTC) | Last {y.min():.0f} bis {y.max():.0f} MW")

kal = kalender(y.index)                                     # Stunde, Wochentag, Monat, Feiertag in Ortszeit
print(kal.loc["2019-03-31 00:00":"2019-03-31 03:00"])        # Zeitumstellung: Ortszeit springt von 2 auf 4 Uhr
check("21 888 Stundenwerte ohne Luecken", len(df) == 21888 and df.index.to_series().diff().dropna().eq(pd.Timedelta("1h")).all())

### 1.2 · Drei Zeitskalen

`CODE` (1 Zeile) · Die Last schwingt gleichzeitig im Jahres-, Wochen- und Tagesrhythmus. Die ersten beiden Bilder sind fertig.
Ergänzen Sie für das dritte Bild das **mittlere Tagesprofil**: der Mittelwert der Last je Stunde (0–23 Uhr Ortszeit), getrennt
nach Werktag, Samstag und Sonntag. Hilfe: `reihe.groupby(stunden).mean()` bildet den Mittelwert je Stunde.

In [ ]:
tr = y.loc[:TRAIN_END] / 1000                             # Trainingszeitraum, in GW
kal_tr = kal.loc[:TRAIN_END]
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
tag = tr.resample("D").mean()                              # Tagesmittel
axs[0].plot(tag, lw=0.8); axs[0].plot(tag.rolling(28, center=True).mean(), lw=2)   # plus gleitendes 4-Wochen-Mittel
axs[0].set_title("Jahr: Tagesmittel"); axs[0].set_ylabel("GW")
axs[1].plot(tr.loc["2019-09-02":"2019-09-29"], lw=0.8); axs[1].set_title("Woche: vier Wochen stuendlich")
axs[1].tick_params(axis="x", rotation=30)
for name, maske in [("Mo–Fr", kal_tr.wochentag < 5), ("Sa", kal_tr.wochentag == 5), ("So", kal_tr.wochentag == 6)]:
    # TODO 1.2: profil = mittlere Last je Stunde (Ortszeit) fuer die Tage in `maske`: tr[maske].groupby(...).mean()
    raise NotImplementedError("TODO 1.2")
    axs[2].plot(profil.index, profil.values, lw=2, label=name)
axs[2].set_title("Tag: mittleres Profil"); axs[2].set_xlabel("Stunde (Ortszeit)"); axs[2].legend()
plt.tight_layout(); plt.show()
check("1.2 Tagesprofil Mo–Fr: Maximum zwischen 10 und 13 Uhr",
      lambda: 10 <= tr[kal_tr.wochentag < 5].groupby(kal_tr.stunde[kal_tr.wochentag < 5]).mean().idxmax() <= 13)

### 1.3 · Autokorrelation selbst berechnen

`CODE` (1 Zeile) · Wie stark hängt die Last von ihrem eigenen Wert vor $k$ Stunden ab?

$$r_k = \frac{\sum_{t=k+1}^{T} (y_t - \bar y)(y_{t-k} - \bar y)}{\sum_{t=1}^{T} (y_t - \bar y)^2}$$

**So liest man die Formel:** Zähler = Summe der Produkte der Abweichungen vom Mittelwert *jetzt* und *vor k Stunden*; Nenner normiert,
sodass $r_0 = 1$. In NumPy: Mit `d = x - x.mean()` ist der Zähler `np.dot(d[:-k], d[k:])` – die Reihe ohne die letzten k Werte mal
die Reihe ohne die ersten k Werte.
**Erwartung (Training):** $r_1 = 0{,}967$ · $r_{12} = -0{,}034$ · $r_{24} = 0{,}761$ · $r_{168} = 0{,}884$.

In [ ]:
def acf(x, k):
    """Autokorrelation der Reihe x (NumPy-Array) bei Verzoegerung k (k >= 1)."""
    d = x - x.mean()                                      # Abweichungen vom Mittelwert
    # TODO 1.3: Zaehler durch Nenner: np.dot(d[:-k], d[k:]) / np.dot(d, d)
    raise NotImplementedError("TODO 1.3")

x_tr = y.loc[:TRAIN_END].values
werte = {k: acf(x_tr, k) for k in [1, 12, 24, 168, 336, 8760]}
print({k: round(v, 3) for k, v in werte.items()})
check("1.3 r_1 = 0,967 und r_24 = 0,761", lambda: nah(acf(x_tr, 1), 0.967, 0.001) and nah(acf(x_tr, 24), 0.761, 0.001))
check("1.3 r_168 = 0,884 (Vorwoche schlaegt Vortag)", lambda: nah(acf(x_tr, 168), 0.884, 0.001))

lags = np.arange(1, 340)
plt.bar(lags, [acf(x_tr, k) for k in lags], width=1); plt.xticks([0, 24, 48, 72, 96, 120, 144, 168, 192, 336])
plt.xlabel("Verzoegerung k in Stunden"); plt.ylabel("r_k"); plt.title("ACF der Last (Training)"); plt.show()
save_log("a1_acf", **{f"r_{k}": v for k, v in werte.items()})

### 1.4 · MSTL-Zerlegung

`AUSFÜHREN` · `MSTL` zerlegt acht Wochen (5.8.–29.9.2019) in Trend + Tagessaison (24 h) + Wochensaison (168 h) + Rest.
**Erwartung:** Varianzanteile Tagessaison ≈ 0,66 · Wochensaison ≈ 0,34 · Trend und Rest je ≈ 0,01.

In [ ]:
fenster = y.loc["2019-08-05":"2019-09-29 23:00"]
sdf = pd.DataFrame({"unique_id": "amprion", "ds": fenster.index, "y": fenster.values})   # Format von statsforecast
sf = StatsForecast(models=[MSTL(season_length=[24, 168])], freq="h").fit(sdf)
zerl = sf.fitted_[0, 0].model_                               # Tabelle: data, trend, seasonal24, seasonal168, remainder
anteile = (zerl.var() / fenster.var()).round(3)
print("Varianzanteile:", anteile.to_dict())
zerl.set_index(fenster.index).plot(subplots=True, figsize=(11, 7), lw=0.8, legend=True); plt.tight_layout(); plt.show()
check("1.4 Tagessaison ≈ 66 %, Wochensaison ≈ 34 % der Varianz",
      nah(anteile["seasonal24"], 0.661, 0.01) and nah(anteile["seasonal168"], 0.335, 0.01))
save_log("a1_mstl", **anteile.to_dict())

**`DENKEN` – Interpretation**

a) Warum ist $r_{168} > r_{24}$? Was vergleicht der Vortag am Montag?
b) Was würde ein Modell, das nur die Tagessaison kennt, systematisch falsch machen?
c) Wo ist der Rest der Zerlegung groß – und was könnte dort passiert sein?

## Aufgabe 2 · Baselines und Backtest

**Block nach „Die Messlatte“ · 15 Min.** · Folien Abschnitt 2 · FPP3 Kap. 5.2, 5.8, 5.10

### 2.1 · Fehlermaße als Funktionen

`CODE` (je 1 Zeile) · Mit $e_t = y_t - \hat y_t$ (Ist minus Prognose):

$$\mathrm{MAE} = \tfrac1n \textstyle\sum_t |e_t| \qquad \mathrm{RMSE} = \sqrt{\tfrac1n \sum_t e_t^2} \qquad
\mathrm{MAPE} = \tfrac{100\,\%}{n} \sum_t \left|\tfrac{e_t}{y_t}\right| \qquad
\mathrm{MASE} = \frac{\mathrm{MAE}}{q},\ q = \tfrac{1}{T-m}\textstyle\sum_{t=m+1}^{T} |y_t - y_{t-m}|$$

`mase` ist vorgegeben (Maßstab $q$ = MAE der saisonal-naiven Prognose mit $m = 168$ im Training).
**Erwartung (Folienbeispiel A):** MAE 1,25 · RMSE 1,50 · MAPE 6,03 % · Maßstab $q$ = 1 063 MW.

In [ ]:
abschnitt("Aufgabe 2")
def mae(ist, prog):
    # TODO 2.1a: mittlerer absoluter Fehler: np.mean(np.abs(ist - prog))
    raise NotImplementedError("TODO 2.1a")

def rmse(ist, prog):
    # TODO 2.1b: Wurzel des mittleren quadratischen Fehlers
    raise NotImplementedError("TODO 2.1b")

def mape(ist, prog):
    # TODO 2.1c: mittlerer absoluter prozentualer Fehler in Prozent
    raise NotImplementedError("TODO 2.1c")

Q_MASE = float(np.mean(np.abs(x_tr[168:] - x_tr[:-168])))    # MAE von "Vorwoche kopieren" im Training
def mase(ist, prog):
    return mae(ist, prog) / Q_MASE

def bewerte(name, prognose):
    """Vergleicht eine Prognose (Series mit Zeitindex) mit der Ist-Last und gibt eine Tabellenzeile zurueck."""
    ist = y.loc[prognose.index].values; p = np.asarray(prognose, dtype=float)
    return {"Modell": name, "MAE": round(mae(ist, p)), "RMSE": round(rmse(ist, p)), "MAPE": round(mape(ist, p), 2),
            "MASE": round(mase(ist, p), 3)}

a, pA = np.array([20, 22, 25, 18.]), np.array([21, 22, 23, 20.])
check("2.1 Folienbeispiel A: MAE 1,25 · RMSE 1,50 · MAPE 6,03 %",
      lambda: nah(mae(a, pA), 1.25, 1e-9) and nah(rmse(a, pA), 1.5, 1e-9) and nah(mape(a, pA), 6.03, 0.01))
print(f"Massstab q fuer MASE: {Q_MASE:.0f} MW")

### 2.2 · Baselines im Backtest

`CODE` (2 Zeilen) · Zwei einfache Prognosen als Messlatte: **naiv** (gleiche Stunde gestern, $\hat y_t = y_{t-24}$) und
**saisonal naiv** (gleiche Stunde vor einer Woche, $\hat y_t = y_{t-168}$). In pandas verschiebt `y.shift(24)` die ganze Reihe um 24
Stunden nach hinten. Schneiden Sie beide auf die Backtest-Stunden `BT_INDEX` zu.
**Erwartung:** naiv MAE 1 562 MW · saisonal naiv 927 MW · Netzbetreiber 466 MW.

In [ ]:
# TODO 2.2: p_naiv = y.shift(24) auf BT_INDEX, p_snaiv = y.shift(168) auf BT_INDEX
raise NotImplementedError("TODO 2.2")
p_tso = df["load_forecast_tso_mw"].astype(float).loc[BT_INDEX]   # Day-ahead-Prognose des Netzbetreibers

ERGEBNISSE = [bewerte("Naiv (Vortag)", p_naiv), bewerte("Saisonal naiv (Vorwoche)", p_snaiv),
              bewerte("Netzbetreiber (Day-ahead)", p_tso)]
display(pd.DataFrame(ERGEBNISSE).set_index("Modell"))
check("2.2 naiv 1 562 MW, saisonal naiv 927 MW", lambda: ERGEBNISSE[0]["MAE"] == 1562 and ERGEBNISSE[1]["MAE"] == 927)
save_log("a2_baselines", ergebnisse=ERGEBNISSE)

### 2.3 · Das Backtest-Gerüst

`LIVE` · Statistische Modelle werden an **jedem** der 28 Ursprünge neu geschätzt – nur mit den Daten bis 23:00 Uhr des Vortags
(`fenster` Stunden zurück) – und sagen dann 24 Stunden voraus. Die Funktion `backtest()` übernimmt das für alle Modelle in Aufgabe 3.
**Probe:** `SeasonalNaive(168)` im Backtest muss genau die Werte von `y.shift(168)` liefern (927 MW).

In [ ]:
def backtest(modelle, fenster=4 * 168):
    """Rollierender Ursprung: fuer jeden Prognosetag mit den letzten `fenster` Stunden schaetzen, 24 Stunden vorhersagen."""
    prognosen = []
    for o in URSPRUENGE:
        # TODO 2.3: Historie bis o - 1 h (letzte `fenster` Werte), ins statsforecast-Format, forecast(h=24), anhaengen
        raise NotImplementedError("TODO 2.3")
    return pd.concat(prognosen).iloc[:, 0]                 # eine Spalte = Prognosen des (ersten) Modells

p_test = backtest([SeasonalNaive(season_length=168)])
check("2.3 backtest() mit SeasonalNaive(168) = y.shift(168): 927 MW", lambda: bewerte("t", p_test)["MAE"] == 927)

fehler_tag = pd.DataFrame({"naiv": (y.loc[BT_INDEX] - p_naiv).abs(), "saisonal naiv": (y.loc[BT_INDEX] - p_snaiv).abs(),
                           "Netzbetreiber": (y.loc[BT_INDEX] - p_tso).abs()}).resample("D").mean()
fehler_tag.plot(marker="o", ylabel="Tages-MAE in MW", title="Fehler je Prognosetag"); plt.show()

**`DENKEN` – Interpretation**

a) Warum ist die naive Prognose montags so schlecht? b) Was passiert am Freitag, 1.11.2019? c) Warum liegt die MASE der
saisonal-naiven Prognose im Backtest unter 1, obwohl sie die Baseline selbst ist?

## Aufgabe 3 · ETS, SARIMA und MSTL

**Block nach „Zwischenstand: Klassische Modelle“ · 15 Min.** · Folien Abschnitt 3 · FPP3 Kap. 8, 9, 12.1

### 3.1 · ETS: Automatik gegen Vorgabe

`CODE` (1 Zeile) · `AutoETS(season_length=24)` wählt die ETS-Variante selbst nach dem kleinsten AICc. Die erste Zelle zeigt, was
die Automatik am ersten Ursprung wählt. Danach: Backtest mit **fest vorgegebener** Variante `model="ANA"` (additiver Fehler,
kein Trend, additive Saison mit Periode 24).
**Erwartung:** Die Automatik wählt ETS(A,Ad,N) – **ohne Saison**. Im Backtest: automatisch 4 778 MW · fest ETS(A,N,A) 1 758 MW.

In [ ]:
abschnitt("Aufgabe 3")
hist = y.loc[:URSPRUENGE[0] - pd.Timedelta(hours=1)].iloc[-4 * 168:]
sf = StatsForecast(models=[AutoETS(season_length=24)], freq="h").fit(pd.DataFrame({"unique_id": "a", "ds": hist.index, "y": hist.values}))
print("AutoETS waehlt am 7.10.2019:", sf.fitted_[0, 0].model_["method"])

p_ets_auto = backtest([AutoETS(season_length=24)])
# TODO 3.1: p_ets = backtest(...) mit AutoETS(season_length=24, model="ANA")
raise NotImplementedError("TODO 3.1")
for name, p in [("AutoETS (Automatik)", p_ets_auto), ("ETS(A,N,A), s = 24", p_ets)]:
    ERGEBNISSE.append(bewerte(name, p)); print(ERGEBNISSE[-1])
check("3.1 ETS(A,N,A): MAE 1 758 MW", lambda: nah(bewerte("t", p_ets)["MAE"], 1758, 2))

### 3.2 · SARIMA im Backtest

`LIVE` · Ordnung $(3,0,0)(2,1,0)_{24}$: einmal saisonal differenzieren ($y_t - y_{t-24}$), dann AR(3) auf die letzten drei Stunden
und saisonales AR(2) auf $t-24$ und $t-48$. Die Ordnung wurde einmal mit AutoARIMA bestimmt und wird fest übernommen (≈ 20 s
Rechenzeit – AutoARIMA an jedem Ursprung würde zu lange dauern). **Erwartung:** 1 620 MW.

In [ ]:
t0 = time.time()
p_sarima = backtest([ARIMA(order=(3, 0, 0), seasonal_order=(2, 1, 0), season_length=24)])
ERGEBNISSE.append(bewerte("SARIMA(3,0,0)(2,1,0)24", p_sarima)); print(ERGEBNISSE[-1], f"| {time.time() - t0:.0f} s")
check("3.2 SARIMA: MAE 1 620 MW", lambda: nah(ERGEBNISSE[-1]["MAE"], 1620, 3))

### 3.3 · MSTL mit zwei Saisons

`CODE` (1 Zeile) · `MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN"))` zieht beide Saisons ab, schreibt den Rest
mit einem ETS ohne Saison fort und addiert die Saisons wieder. Fenster **8 Wochen** (`fenster=8 * 168`), damit die Wochensaison
mehrfach vorkommt. **Erwartung:** 673 MW – deutlich besser als saisonal naiv (927 MW).

In [ ]:
# TODO 3.3: p_mstl = backtest([MSTL(...)], fenster=8 * 168)
raise NotImplementedError("TODO 3.3")
ERGEBNISSE.append(bewerte("MSTL + ETS", p_mstl)); print(ERGEBNISSE[-1])
check("3.3 MSTL: MAE 673 MW", lambda: nah(bewerte("t", p_mstl)["MAE"], 673, 3))

w = slice("2019-10-28", "2019-11-03 23:00")                 # eine Woche mit Allerheiligen
pd.DataFrame({"Ist": y.loc[w], "ETS(A,N,A)": p_ets.loc[w], "SARIMA": p_sarima.loc[w], "MSTL": p_mstl.loc[w]}).plot(
    style=["-", ":", "--", "-."], ylabel="MW", title="Backtest-Woche 28.10.–3.11.2019"); plt.show()
display(pd.DataFrame(ERGEBNISSE).set_index("Modell"))
save_log("a3_statistik", ergebnisse=ERGEBNISSE)

**`DENKEN` – Interpretation**

a) Warum ist ETS(A,N,A) sogar schlechter als die naive Prognose? b) Warum braucht MSTL ein längeres Fenster? c) Was lehrt die
Wahl der Automatik in 3.1 über automatische Modellwahl?

## Aufgabe 4 · LightGBM und LSTM

**Block nach „Ein LSTM für 24 Stunden“ · 15 Min.** · Folien Abschnitt 4 · Géron Kap. 15

### 4.1 · Merkmale leckagefrei bauen

`CODE` (2 Zeilen) · Jede Stunde wird eine Tabellenzeile: Kalender (Ortszeit) plus Lags. Ergänzen Sie in `merkmale()` die
**Lags 24, 48, 168 und 336 Stunden** (Spaltennamen `lag_24`, …) und das **Mittel der 24 Stunden bis t − 24**
(`mittel_24_lag24` = gleitendes Mittel über 24 Werte von `y.shift(24)`).
**Wichtig:** Kein Lag unter 24 h und das gleitende Mittel auf `y.shift(24)`, nicht auf `y` – sonst steckt die Zielstunde im Merkmal.

In [ ]:
abschnitt("Aufgabe 4")
def merkmale(y):
    X = kalender(y.index)                                  # stunde, wochentag, monat, feiertag
    #<TODO 4.1 | Lags 24, 48, 168, 336 als Spalten lag_<l>; mittel_24_lag24 = y.shift(24).rolling(24).mean()>
    for l in [24, 48, 168, 336]:
        X[f"lag_{l}"] = y.shift(l)                         # Wert vor l Stunden (l >= 24 -> zum Ursprung bekannt)
    X["mittel_24_lag24"] = y.shift(24).rolling(24).mean()  # Niveau des Vortags
    #</TODO>
    return X

X = merkmale(y)
maske_train = (X.index <= TRAIN_END) & X.notna().all(axis=1)   # Trainingszeilen mit vollstaendigen Lags
print(f"{maske_train.sum()} Trainingszeilen, {X.shape[1]} Merkmale: {list(X.columns)}")
# Leckage-Probe: aendert man y ab dem Ursprung, duerfen sich die Merkmale der Zielstunden nicht aendern
y_test = y.copy(); y_test.loc["2019-10-07":] += 1e6
check("4.1 Merkmale leckagefrei (Zielstunden haengen nur von Werten bis 23:00 des Vortags ab)",
      lambda: merkmale(y_test).loc["2019-10-07 00:00":"2019-10-07 23:00"].equals(X.loc["2019-10-07 00:00":"2019-10-07 23:00"]))
check("4.1 neun Merkmale", lambda: X.shape[1] == 9)

### 4.2 · LightGBM trainieren und deuten

`CODE` (2 Zeilen) · `LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=0, verbose=-1)` auf den
Trainingszeilen anpassen (`fit(X[maske_train], y[maske_train])`) und für `BT_INDEX` vorhersagen. Weil alle Lags ≥ 24 h sind, ist
jede Zeile direkt eine gültige Day-ahead-Prognose – keine Schleife über Ursprünge nötig.
**Erwartung:** MAE 488 MW (Netzbetreiber 466 MW), Training < 1 s.

In [ ]:
t0 = time.time()
# TODO 4.2: gbm anlegen und trainieren; p_gbm = pd.Series(gbm.predict(X.loc[BT_INDEX]), index=BT_INDEX)
raise NotImplementedError("TODO 4.2")
print(f"Training + Vorhersage: {time.time() - t0:.1f} s")
ERGEBNISSE.append(bewerte("LightGBM", p_gbm)); print(ERGEBNISSE[-1])
check("4.2 LightGBM: MAE 488 MW", lambda: nah(bewerte("t", p_gbm)["MAE"], 488, 5))
pd.Series(gbm.feature_importances_, index=X.columns).sort_values().plot.barh(
    title="Merkmalswichtigkeit (Anzahl Splits)", figsize=(6, 3.5)); plt.show()

### 4.3 · Ein LSTM für 24 Stunden

`LIVE` · Eingabe: die letzten **168 Stunden** mit vier Kanälen je Stunde – Last (standardisiert), $\sin$ und $\cos$ der Stunde,
Wochenend-Flag. Ein LSTM mit 32 Zuständen fasst die Woche zusammen, eine lineare Schicht gibt **24 Werte auf einmal** aus.
Training: Fenster alle 3 Stunden aus dem Trainingszeitraum, MSE-Verlust, Adam, 8 Epochen – die Trainingsschleife aus Woche 1.
**Erwartung:** 5 656 Parameter, Verlust fällt von ≈ 0,72 auf ≈ 0,11, MAE ≈ 745 MW, Training ≈ 10 s.

In [ ]:
torch.manual_seed(0)
mu, sd = y.loc[:TRAIN_END].mean(), y.loc[:TRAIN_END].std()   # Standardisierung nur mit Trainingsstatistik
z = ((y - mu) / sd).values.astype(np.float32)
h_sin = np.sin(2 * np.pi * kal.stunde / 24).values           # Stunde als Kreis: 23 Uhr liegt neben 0 Uhr
h_cos = np.cos(2 * np.pi * kal.stunde / 24).values
we = (kal.wochentag >= 5).values.astype(float)               # Wochenende ja/nein
F = np.stack([z, h_sin, h_cos, we], axis=1).astype(np.float32)   # 21 888 x 4
ENC, H = 168, 24

def fenster(starts):
    """Fuer jeden Startindex s: Eingabe F[s-168 : s] (168 x 4) und Ziel z[s : s+24] (24 Werte)."""
    Xs = np.stack([F[s - ENC:s] for s in starts]); Ys = np.stack([z[s:s + H] for s in starts])
    return torch.tensor(Xs), torch.tensor(Ys)

ende = y.index.get_loc(TRAIN_END)                            # Position der letzten Trainingsstunde
Xtr, Ytr = fenster(np.arange(ENC, ende - H + 2, 3))          # Ziel endet spaetestens am 30.9. 23:00

class LSTMPrognose(torch.nn.Module):
    def __init__(self, n_in=4, hidden=32, H=24):
        super().__init__()
        self.lstm = torch.nn.LSTM(n_in, hidden, batch_first=True)   # laeuft ueber die 168 Schritte
        self.kopf = torch.nn.Linear(hidden, H)                     # letzter Zustand -> 24 Stunden
    def forward(self, x):
        _, (h, _) = self.lstm(x)                                   # h: letzter verborgener Zustand
        return self.kopf(h[-1])

modell = LSTMPrognose()
print("Parameter:", sum(p.numel() for p in modell.parameters()), "| Trainingsfenster:", len(Xtr))
opt, lossf = torch.optim.Adam(modell.parameters(), lr=3e-3), torch.nn.MSELoss()
t0 = time.time()
for epoche in range(8):                                      # Trainingsschleife wie in Woche 1
    modell.train(); perm = torch.randperm(len(Xtr)); summe = 0.0
    for i in range(0, len(Xtr), 128):
        b = perm[i:i + 128]
        opt.zero_grad(); verlust = lossf(modell(Xtr[b]), Ytr[b]); verlust.backward(); opt.step()
        summe += verlust.item() * len(b)
    print(f"Epoche {epoche + 1}: Verlust {summe / len(Xtr):.4f} | {time.time() - t0:.1f} s")

modell.eval()
starts = [y.index.get_loc(o) for o in URSPRUENGE]            # Ursprung = erste Stunde des Prognosetags
with torch.no_grad():
    P = modell(fenster(starts)[0]).numpy() * sd + mu          # zurueck in MW
p_lstm = pd.Series(P.ravel(), index=BT_INDEX)
ERGEBNISSE.append(bewerte("LSTM", p_lstm)); print(ERGEBNISSE[-1])
check("4.3 LSTM: 5 656 Parameter, MAE unter 927 MW (saisonal naiv)",
      lambda: sum(p.numel() for p in modell.parameters()) == 5656 and bewerte("t", p_lstm)["MAE"] < 927)
pd.DataFrame({"Ist": y.loc[w], "LightGBM": p_gbm.loc[w], "LSTM": p_lstm.loc[w]}).plot(style=["-", "--", ":"], ylabel="MW"); plt.show()
save_log("a4_ml", ergebnisse=ERGEBNISSE[-2:])

**`DENKEN` – Interpretation**

a) Welche Information hat LightGBM, die das LSTM nicht hat? b) Würde längeres Training dem LSTM viel helfen – was ist der eigentliche
Engpass? c) Die Merkmalswichtigkeit zählt Splits: Warum steht `feiertag` ganz unten, obwohl es an Allerheiligen entscheidend ist?

## Aufgabe 5 · TFT und Modellwahl

**Block nach „Zwischenstand: ML und Deep Learning“ · 10 Min.** · Folien Abschnitt 4 · Lim et al. (2021)

### 5.1 · Den TFT befragen

`LIVE` · Der Temporal Fusion Transformer (64 319 Parameter) wurde auf 2018-01 bis 2019-08 trainiert (September = Validierung,
≈ 7 Min. auf der CPU eines Apple-M4-Pro-Notebooks). Wir laden nur den Checkpoint und sagen für die 28 Tage vorher: je Stunde drei Quantile
(10 %, 50 %, 90 %). Dazu die Deutungshilfen: Attention über die 168 Stunden und die Gewichte der Variablenauswahl.
**Erwartung:** Median-MAE 598 MW, Inferenz ≈ 3 s.

In [ ]:
abschnitt("Aufgabe 5")
NUTZE_REFERENZ = False                     # True: mitgelieferte Vorhersagen laden (falls der TFT nicht laeuft)
try:
    if NUTZE_REFERENZ: raise RuntimeError("Referenz gewaehlt")
    import logging; logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    from pytorch_forecasting import TemporalFusionTransformer
    tft = TemporalFusionTransformer.load_from_checkpoint(TFT_CKPT, map_location="cpu"); tft.eval()
    d = df.reset_index()[["ds", "load_mw"]].copy()          # dieselbe Aufbereitung wie beim Training
    d = pd.concat([d, kalender(d.ds).reset_index(drop=True)], axis=1)
    d["time_idx"] = np.arange(len(d)); d["serie"] = "amprion"; d["load_mw"] = d["load_mw"].astype(float)
    for c in ["stunde", "wochentag", "monat", "feiertag"]: d[c] = d[c].astype(str)
    t0 = time.time(); Q, ATT, VI = [], [], []
    for o in URSPRUENGE:
        i0 = d.index[d.ds == o][0]
        fen = d.iloc[i0 - 168:i0 + 24]                        # 168 h Vergangenheit + 24 h bekannte Zukunft (Kalender)
        roh = tft.predict(fen, mode="raw", return_x=True, trainer_kwargs=dict(accelerator="cpu", logger=False, enable_progress_bar=False))
        Q.append(pd.DataFrame(roh.output.prediction[0].numpy(), columns=["q10", "q50", "q90"], index=fen.ds.iloc[168:]))
        deut = tft.interpret_output(roh.output, reduction="sum")
        ATT.append(deut["attention"].numpy()); VI.append(deut["decoder_variables"].numpy())
    p_tft = pd.concat(Q); print(f"Inferenz: {time.time() - t0:.1f} s")
    att = np.mean(ATT, axis=0); att /= att.sum()
    vi = pd.Series(np.mean(VI, axis=0), index=tft.decoder_variables); vi /= vi.sum()
except Exception as e:
    print(f"TFT nicht nutzbar ({type(e).__name__}: {e}) -> Referenzvorhersagen aus results_referenz/")
    p_tft = pd.read_csv(REFDIR / "tft_backtest.csv", index_col=0, parse_dates=True)
    att = np.load(REFDIR / "tft_attention.npy"); vi = pd.Series(json.load(open(REFDIR / "tft_variablen.json")))
p_tft.index = pd.DatetimeIndex(p_tft.index); p_tft.index.name = "ds"
ERGEBNISSE.append(bewerte("TFT (Median)", p_tft["q50"])); print(ERGEBNISSE[-1])
check("5.1 TFT: Median-MAE 598 MW", lambda: nah(ERGEBNISSE[-1]["MAE"], 598, 5))

fig, axs = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [1.7, 1]})
axs[0].bar(168 - np.arange(len(att)), att * 100, width=1); axs[0].invert_xaxis()
axs[0].set_xlabel("Stunden vor dem Ursprung"); axs[0].set_ylabel("Anteil in %"); axs[0].set_title("Attention")
vi.sort_values().plot.barh(ax=axs[1], title="Variablenwahl Decoder (bekannte Zukunft)"); plt.tight_layout(); plt.show()
tage = [att[(168 - 24 * (k + 1)):(168 - 24 * k)].sum() for k in range(7)]   # Anteil je Tag zurueck
print("Attention-Anteil je Tag zurueck (1 = letzte 24 h):", [f"{t:.0%}" for t in tage])

### 5.2 · Das Band prüfen

`CODE` (2 Zeilen) · Ein 10–90-%-Band sollte im Mittel **80 %** der Ist-Werte enthalten (*Abdeckung*). Berechnen Sie die Abdeckung
(Anteil der Stunden mit $q_{0,1} \le y \le q_{0,9}$) und den Pinball-Verlust je Quantil:

$$L_\tau(y, q) = \begin{cases} \tau\,(y - q) & y \ge q \\ (1 - \tau)\,(q - y) & y < q \end{cases} \qquad\text{in NumPy:}\quad
e = y - q,\ \ L = \max(\tau e,\ (\tau - 1) e)$$

**Erwartung:** Abdeckung 53 % · Pinball $q_{0,1}$ 195 · $q_{0,5}$ 299 · $q_{0,9}$ 123 MW.

In [ ]:
ist = y.loc[p_tft.index]
# TODO 5.2a: abdeckung = Anteil der Stunden mit q10 <= ist <= q90 (Mittelwert einer booleschen Series)
raise NotImplementedError("TODO 5.2a")
def pinball(ist, q, tau):
    e = ist - q
    # TODO 5.2b: mittlerer Pinball-Verlust: np.mean(np.maximum(tau * e, (tau - 1) * e))
    raise NotImplementedError("TODO 5.2b")
pb = {f"q{int(t * 100)}": round(pinball(ist, p_tft[f"q{int(t * 100)}"], t)) for t in (0.1, 0.5, 0.9)}
print(f"Abdeckung des 10–90-%-Bands: {abdeckung:.1%} (Soll 80 %) | Pinball: {pb}")
check("5.2 Abdeckung 53 %", lambda: nah(abdeckung, 0.528, 0.01))
check("5.2 Pinball q50 = halber MAE des Medians", lambda: nah(pinball(ist, p_tft["q50"], 0.5), mae(ist.values, p_tft["q50"].values) / 2, 1e-6))
w = slice("2019-10-28", "2019-11-03 23:00")
plt.fill_between(p_tft.loc[w].index, p_tft.loc[w, "q10"], p_tft.loc[w, "q90"], alpha=0.3, label="10–90 %")
plt.plot(p_tft.loc[w, "q50"], "--", label="Median"); plt.plot(y.loc[w], label="Ist"); plt.legend(); plt.ylabel("MW"); plt.show()
save_log("a5_tft", mae_median=ERGEBNISSE[-1]["MAE"], abdeckung=abdeckung, pinball=pb)

### 5.3 · Vergleich, Empfehlung, Endtest

`AUSFÜHREN` + `DENKEN` · Die Tabelle fasst alle Modelle im selben Backtest zusammen. Schreiben Sie darunter **drei Sätze**: Welches
Modell würden Sie dem Netzbetreiber empfehlen – und warum (Genauigkeit, Aufwand, Unsicherheit, Erklärbarkeit)? Erst **danach**
setzen Sie `ENDTEST_FREIGEBEN = True` und rechnen den Endtest (Dezember 2019) **genau einmal** für Ihr Modell.
**Erwartung Endtest:** LightGBM 838 MW · Netzbetreiber 821 MW · saisonal naiv 1 886 MW.

In [ ]:
tab = pd.DataFrame(ERGEBNISSE).drop_duplicates("Modell", keep="last").set_index("Modell").sort_values("MAE")
display(tab)
save_log("a5_vergleich", tabelle=tab.reset_index().to_dict(orient="records"))

**Meine Empfehlung (3 Sätze):**

*…*

In [ ]:
ENDTEST_FREIGEBEN = False                  # erst nach der Empfehlung auf True setzen – und nur einmal ausfuehren
if ENDTEST_FREIGEBEN:
    endtest = [bewerte("LightGBM", pd.Series(gbm.predict(X.loc[ENDTEST]), index=ENDTEST)),
               bewerte("Saisonal naiv", y.shift(168).loc[ENDTEST]),
               bewerte("Netzbetreiber", df["load_forecast_tso_mw"].astype(float).loc[ENDTEST])]
    display(pd.DataFrame(endtest).set_index("Modell"))
    check("5.3 Endtest Dezember: LightGBM 838 MW", lambda: nah(endtest[0]["MAE"], 838, 5))
    save_log("a5_endtest", ergebnisse=endtest)

**`DENKEN`:** a) Warum ist das Band zu schmal? b) Warum darf der Endtest nur einmal gerechnet werden? c) Warum ist der Endtest-MAE
fast doppelt so hoch wie im Backtest?

## Aufgabe 6 · Drift und Anomalien

**Block nach „Zwischenstand: Anomalien und Drift“ · 10 Min.** · Folien Abschnitt 5 · FPP3 5.4 · Liu et al. (2008)

### 6.1 · Das Modell durch den Lockdown schicken

`CODE` (2 Zeilen) · Das LightGBM aus Aufgabe 4 bleibt **unverändert** (Stand 30.9.2019) und sagt vom 1.10.2019 bis 30.6.2020 voraus.
Berechnen Sie das Residuum $e_t = y_t - \hat y_t$ und je Monat den **MAE** (Mittel von $|e|$) und den **Bias** (Mittel von $e$).
Hilfe: `reihe.resample("MS").mean()` bildet Monatsmittel.
**Erwartung Bias:** Jan −306 · Mär −350 · Apr −302 · Mai −321 MW · MAE Apr 613 MW (Netzbetreiber 1 075 MW).

In [ ]:
abschnitt("Aufgabe 6")
DRIFT = pd.date_range("2019-10-01", "2020-06-30 23:00", freq="h")
res = y.loc[DRIFT] - gbm.predict(X.loc[DRIFT])                         # Residuum LightGBM
res_tso = y.loc[DRIFT] - df["load_forecast_tso_mw"].loc[DRIFT]           # Residuum Netzbetreiber
# TODO 6.1: monat = DataFrame mit Spalten MAE (res.abs() Monatsmittel), Bias (res Monatsmittel), MAE_TSO
raise NotImplementedError("TODO 6.1")
display(monat)
check("6.1 Bias April 2020 ≈ −302 MW, MAE April ≈ 613 MW",
      lambda: nah(monat.loc["2020-04-01", "Bias"], -302, 5) and nah(monat.loc["2020-04-01", "MAE"], 613, 5))
woche = pd.DataFrame({"Bias LightGBM": res.resample("W-SUN").mean(), "Bias Netzbetreiber": res_tso.resample("W-SUN").mean()})
ax = woche.plot(marker="o", ylabel="Wochen-Bias in MW", title="Vorzeichen des Fehlers")
ax.axvspan(pd.Timestamp("2020-03-16"), pd.Timestamp("2020-05-10"), alpha=0.1, color="orange"); ax.axhline(0, color="grey", lw=0.8); plt.show()
save_log("a6_drift", monat=monat.reset_index(names="monat").astype({"monat": str}).to_dict(orient="records"))

### 6.2 · Isolation Forest auf Tagesmerkmalen

`CODE` (2 Zeilen) · Je Tag drei Merkmale des Residuums: Mittelwert, Standardabweichung, größter Betrag. Ein Isolation Forest
(`IsolationForest(n_estimators=200, contamination=0.05, random_state=0)`) markiert die 5 % auffälligsten Tage. `fit(F)`,
dann `predict(F)` (−1 = Anomalie) und `score_samples(F)` (je kleiner, desto auffälliger).
**Erwartung:** 14 von 274 Tagen; die fünf mit höchstem Score: 24.12.2019, 1.1.2020, 1.12.2019, 24.2.2020, 3.12.2019.

In [ ]:
tag = res.resample("D")
F = pd.DataFrame({"mittel": tag.mean(), "streuung": tag.std(), "max_abs": tag.apply(lambda s: s.abs().max())})
# TODO 6.2: iso anlegen und fitten; F["anomalie"] = (iso.predict(...) == -1); F["score"] = -iso.score_samples(...)
raise NotImplementedError("TODO 6.2")
top = F[F.anomalie].sort_values("score", ascending=False)
display(top.round(0).head(14))
check("6.2 14 Anomalien, auffaelligster Tag 24.12.2019",
      lambda: int(F.anomalie.sum()) == 14 and top.index[0] == pd.Timestamp("2019-12-24"))
ax = (F["mittel"] / 1000).plot(lw=1, ylabel="mittleres Residuum in GW", title="Tagesresiduum mit Anomalien")
ax.scatter(top.index, top["mittel"] / 1000, color="orange", zorder=3); plt.show()
save_log("a6_anomalien", tage=[str(t.date()) for t in top.index], anzahl=int(F.anomalie.sum()))

**`DENKEN` – Interpretation**

a) LightGBM bleibt im Lockdown beim MAE stabil, der Netzbetreiber nicht – warum? b) Warum findet der Isolation Forest den Lockdown
kaum, Heiligabend aber sofort? c) Welche zwei Merkmale würden Sie als Nächstes ergänzen?

## Laufzeiten und Prüfungen

`AUSFÜHREN` · Reine Rechenzeit je Aufgabe und Übersicht aller Prüfungen.

In [ ]:
abschnitt("Abschluss")
display(pd.DataFrame({"Rechenzeit (s)": RUNTIME}).round(1))
pflicht = sum(v for k, v in RUNTIME.items() if k.startswith("Aufgabe"))
print(f"Pflichtteil (Aufgaben 0-6): {pflicht / 60:.1f} Min. | Threads: {torch.get_num_threads()}")
save_log("laufzeiten", runtime_s=RUNTIME, pflicht_min=pflicht / 60, cpu_kerne=os.cpu_count())
print(f"Pruefungen: {sum(CHECKS.values())} ✅ / {sum(not v for v in CHECKS.values())} ❌")
for name, ok in CHECKS.items():
    if not ok:
        print("  ❌", name)